# DB stuff

A tour of the schema and of the data. Nothing here writes.

It runs on `staging/_template` by default, which is the only staging directory
in the repository. Uncomment the second line in the next cell to load every
source directory instead.

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB
from loader import load, load_all
from loader.load import STAGING # the repository's staging directory

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

db = ProbeDB(":memory:", create=True)

load(db, STAGING / "_template", source="template")
# load_all(db, STAGING) # every source directory instead

db.counts()

,table,rows
0,compound,3
1,chembl,3
2,uniprot,8
3,target,6
4,target_uniprot,9
5,bioactivity_source,10
6,bioactivity_group,8
7,bioactivity,13


## Tables

In [2]:
from probedb import schema

columns = []
for table in schema.TABLES:
    for row in db.conn.execute(f"PRAGMA table_info({table})"):
        columns.append({"table": table, "column": row[1], "type": row[2],
                        "not_null": bool(row[3]), "primary_key": bool(row[5])})

pd.DataFrame(columns)

,table,column,type,not_null,primary_key
0,compound,inchikey,VARCHAR(27),False,True
1,compound,smiles,TEXT,False,False
2,compound,name,VARCHAR(255),False,False
3,chembl,chembl_id,VARCHAR(20),False,True
4,chembl,inchikey,VARCHAR(27),True,False
5,uniprot,uniprot_id,VARCHAR(20),False,True
6,uniprot,entrez_gene,VARCHAR(50),False,False
7,uniprot,hgnc,VARCHAR(50),False,False
8,uniprot,species,VARCHAR(100),False,False
9,target,target_id,INTEGER,False,True


In [3]:
db.table("compound")

,inchikey,smiles,name
0,XQVVPGYIWAGRNI-JOCHJYFZSA-N,N1([C@@H](C(=O)N(c2c1nc(nc2)Nc3c(cc(cc3)...,BI-2536
1,DNVXATUJJDPFDM-KRWDZBQOSA-N,Cc1sc2c(c1C)C(c1ccc(Cl)cc1)=N[C@@H](CC(=...,(+)-JQ1
2,FDLYAMZZIXQODN-UHFFFAOYSA-N,O=C(c1cc(Cc2n[nH]c(=O)c3ccccc23)ccc1F)N1...,Olaparib


In [4]:
# chembl maps a ChEMBL id to an InChIKey, one row per id
db.table("chembl")

,chembl_id,inchikey
0,CHEMBL513909,XQVVPGYIWAGRNI-JOCHJYFZSA-N
1,CHEMBL1957266,DNVXATUJJDPFDM-KRWDZBQOSA-N
2,CHEMBL521686,FDLYAMZZIXQODN-UHFFFAOYSA-N


In [5]:
db.table("uniprot").head(12)

,uniprot_id,entrez_gene,hgnc,species
0,P53350,None,PLK1,Homo sapiens
1,O60885,None,BRD4,Homo sapiens
2,P25440,None,BRD2,Homo sapiens
3,P09874,None,PARP1,Homo sapiens
4,Q9UGN5,None,PARP2,Homo sapiens
5,Q9Y6F1,None,PARP3,Homo sapiens
6,P06493,None,CDK1,Homo sapiens
7,P14635,None,CCNB1,Homo sapiens


In [6]:
db.table("target")

,target_id,type,name
0,1,protein,Serine/threonine-protein kinase PLK1
1,2,protein,Bromodomain-containing protein 4
2,3,protein,Bromodomain-containing protein 2
3,4,protein,Poly [ADP-ribose] polymerase 1
4,5,family,"PARP 1, 2 and 3"
5,6,complex,Cyclin-dependent kinase 1/cyclin B1


In [7]:
db.table("target_flat")

,target_id,type,name,uniprot_id,hgnc,species
0,1,protein,Serine/threonine-protein kinase PLK1,P53350,PLK1,Homo sapiens
1,2,protein,Bromodomain-containing protein 4,O60885,BRD4,Homo sapiens
2,3,protein,Bromodomain-containing protein 2,P25440,BRD2,Homo sapiens
3,4,protein,Poly [ADP-ribose] polymerase 1,P09874,PARP1,Homo sapiens
4,5,family,"PARP 1, 2 and 3",P09874,PARP1,Homo sapiens
5,5,family,"PARP 1, 2 and 3",Q9UGN5,PARP2,Homo sapiens
6,5,family,"PARP 1, 2 and 3",Q9Y6F1,PARP3,Homo sapiens
7,6,complex,Cyclin-dependent kinase 1/cyclin B1,P06493,CDK1,Homo sapiens
8,6,complex,Cyclin-dependent kinase 1/cyclin B1,P14635,CCNB1,Homo sapiens


In [8]:
db.table("bioactivity").head(12)

,id,inchikey,target_id,moa,bioactivity_type,relation,value,unit,assay_type,assay_description,cell_line,concentration,concentration_unit,source_id,source_xref
0,1,XQVVPGYIWAGRNI-JOCHJYFZSA-N,1,inhibitor,IC50,=,0.83,nM,biochemical,kinase activity assay,NaN,None,None,1,probe:BI-2536
1,2,XQVVPGYIWAGRNI-JOCHJYFZSA-N,1,inhibitor,pIC50,=,10.08,-log(M),biochemical,NaN,NaN,None,None,2,activity:138064
2,3,XQVVPGYIWAGRNI-JOCHJYFZSA-N,1,inhibitor,IC50,=,1.10,nM,biochemical,kinase activity assay,NaN,None,None,3,PLK1-BIO-0042
3,4,XQVVPGYIWAGRNI-JOCHJYFZSA-N,1,inhibitor,EC50,=,12.00,nM,cell,antiproliferation,HeLa,None,None,3,PLK1-CELL-0007
4,5,XQVVPGYIWAGRNI-JOCHJYFZSA-N,2,inhibitor,IC50,=,1.20,nM,biochemical,NaN,NaN,None,None,4,NaN
5,6,XQVVPGYIWAGRNI-JOCHJYFZSA-N,6,,IC50,>,10000.00,nM,biochemical,kinase counter-screen,NaN,None,None,5,NaN
6,7,DNVXATUJJDPFDM-KRWDZBQOSA-N,2,inhibitor,EC50,=,144.50,nM,cell,NaN,MV4-11,None,None,6,NaN
7,8,DNVXATUJJDPFDM-KRWDZBQOSA-N,3,inhibitor,IC50,=,120.20,nM,biochemical,NaN,NaN,None,None,7,NaN
8,9,DNVXATUJJDPFDM-KRWDZBQOSA-N,3,inhibitor,IC50,=,141.30,nM,biochemical,NaN,NaN,None,None,7,NaN
9,10,DNVXATUJJDPFDM-KRWDZBQOSA-N,3,inhibitor,Kd,=,10.96,nM,binding,NaN,NaN,None,None,8,NaN


## Complexes

`target_flat` is the flat, one-row-per-accession shape. A complex repeats its
`target_id`, once per subunit.

In [9]:
db.read('''
    SELECT * FROM target_flat
     WHERE target_id IN (SELECT target_id FROM target_uniprot WHERE uniprot_id = ?)
     ORDER BY target_id, uniprot_id
''', "P09874")

,target_id,type,name,uniprot_id,hgnc,species
0,4,protein,Poly [ADP-ribose] polymerase 1,P09874,PARP1,Homo sapiens
1,5,family,"PARP 1, 2 and 3",P09874,PARP1,Homo sapiens
2,5,family,"PARP 1, 2 and 3",Q9UGN5,PARP2,Homo sapiens
3,5,family,"PARP 1, 2 and 3",Q9Y6F1,PARP3,Homo sapiens


## One compound

Compounds are searchable by name as well as by InChIKey.

In [10]:
BI_3802 = "GXTJETQFYHZHNB-GASCZTMLSA-N"

db.bioactivities(compound="BI-2536")[
    ["target", "moa", "bioactivity_type", "relation", "value", "unit",
     "assay_type", "cell_line", "source_db", "source"]]

,target,moa,bioactivity_type,relation,value,unit,assay_type,cell_line,source_db,source
0,Serine/threonine-protein kinase PLK1,inhibitor,IC50,=,0.83,nM,biochemical,NaN,opnMe,NaN
1,Serine/threonine-protein kinase PLK1,inhibitor,pIC50,=,10.08,-log(M),biochemical,NaN,Probes & Drugs,PMID:33539089
2,Serine/threonine-protein kinase PLK1,inhibitor,IC50,=,1.10,nM,biochemical,NaN,in-house,assay report 2024-03
3,Serine/threonine-protein kinase PLK1,inhibitor,EC50,=,12.00,nM,cell,HeLa,in-house,assay report 2024-03
4,Bromodomain-containing protein 4,inhibitor,IC50,=,1.20,nM,biochemical,NaN,literature,PMID:32088495
5,Cyclin-dependent kinase 1/cyclin B1,,IC50,>,10000.00,nM,biochemical,NaN,literature,PMID:17291758


## Units

Units are kept as the source reported them, nothing is converted. Probes & Drugs
holds no raw concentrations at all - all 982K of its measured rows are pre-logged, so its rows arrive as pIC50 on
-log(M).

So a comparison has to say which unit it is comparing on.

In [11]:
activity = db.bioactivities()

activity.groupby(["bioactivity_type", "unit"], as_index=False).size()

,bioactivity_type,unit,size
0,EC50,nM,2
1,IC50,nM,8
2,Kd,nM,1
3,Kd,uM,1
4,pIC50,-log(M),1


## Repeated measurements

Nothing is averaged on load, so the spread is still there to look at.

In [12]:
groups = (
    activity
    .groupby(["compound", "target", "bioactivity_type", "unit"], as_index=False)
    .agg(n=("value", "size"), minimum=("value", "min"), maximum=("value", "max"),
         median=("value", "median"))
    .sort_values("n", ascending=False)
)

groups.head(10)

,compound,target,bioactivity_type,unit,n,minimum,maximum,median
0,(+)-JQ1,Bromodomain-containing protein 2,IC50,nM,2,120.20,141.30,130.750
6,BI-2536,Serine/threonine-protein kinase PLK1,IC50,nM,2,0.83,1.10,0.965
1,(+)-JQ1,Bromodomain-containing protein 2,Kd,nM,1,10.96,10.96,10.960
2,(+)-JQ1,Bromodomain-containing protein 4,EC50,nM,1,144.50,144.50,144.500
3,BI-2536,Bromodomain-containing protein 4,IC50,nM,1,1.20,1.20,1.200
4,BI-2536,Cyclin-dependent kinase 1/cyclin B1,IC50,nM,1,10000.00,10000.00,10000.000
5,BI-2536,Serine/threonine-protein kinase PLK1,EC50,nM,1,12.00,12.00,12.000
7,BI-2536,Serine/threonine-protein kinase PLK1,pIC50,-log(M),1,10.08,10.08,10.080
8,Olaparib,"PARP 1, 2 and 3",IC50,nM,1,20.89,20.89,20.890
9,Olaparib,Poly [ADP-ribose] polymerase 1,IC50,nM,1,0.10,0.10,0.100
